In [12]:
import pandas as pd
import numpy as np
import re

from pathlib import Path
from glob import glob

# Load Metadata

In [ ]:

metadata = pd.read_csv(
    "../../data/metadata/station_metadata.csv",
    skipinitialspace=True,
    quotechar='"'
)

metadata.columns = metadata.columns.str.strip()

# Clean string columns
for col in metadata.columns:
    if metadata[col].dtype == "object":
        metadata[col] = (
            metadata[col]
            .astype(str)
            .str.strip()
            .str.replace('"', '', regex=False)
        )

print("Metadata Shape:", metadata.shape)
print("Stations:", len(metadata))

metadata[["station_id", "station_name"]].head()

Metadata Shape: (17, 12)
Stations: 17


,station_id,station_name
0,WB001,Avidipta Housing Complex
1,WB002,"Ballygunge Campus, C.U"
2,WB003,Bethune College
3,WB004,Dhapa Lock Pumping Station
4,WB005,"East Calcutta Girls College, Lake Town"


# Load All Station Files



In [17]:
# =====================================================
# Load All Station Files
# =====================================================

files = glob("../../data/raw/*.xlsx")

print("Files found:", len(files))

all_dfs = []

# Explicit mapping to avoid metadata inconsistencies
station_mapping = {
    "Avidipta Housing Complex": "Avidipta Housing Complex",
    "Ballygunge Campus, C.U": "Ballygunge Campus, C.U",
    "Bethune College": "Bethune College",
    "Dhapa Lock Pumping Station": "Dhapa Lock Pumping Station",
    "East Calcutta Girls College, Lake Town": "East Calcutta Girls College, Lake Town",
    "Flora Fountain": "Flora Fountain",
    "Fortis Hospital": "Fortis Hospital",
    "Lady Brabourne College": "Lady Brabourne College",
    "Leather Complex": "Leather Complex",
    "Lorreto College": "Lorreto College",
    "Madhyamgram Municipality": "Madhyamgram Municipality",
    "Maulana Azad Collage": "Maulana Azad Collage",
    "Presidency University": "Presidency University",
    "Sarojini Naidu College for Women": "Sarojini Naidu College for Women",
    "Sarsuna College": "Sarsuna College",
    "Urbana Housing Complex": "Urbana Housing Complex",
    "VIVEKANANDA COLLEGE FOR WOMEN, BARISHA": "VIVEKANANDA COLLEGE FOR WOMEN, BARISHA"
}

for file in sorted(files):

    station_name = (
        Path(file)
        .stem
        .replace("-Jan 1, 2022-May 30, 2026", "")
        .strip()
    )

    metadata_name = station_mapping.get(
        station_name,
        station_name
    )

    station_info = metadata[
        metadata["station_name"].str.strip()
        == metadata_name
    ]

    if station_info.empty:
        print(f"WARNING: Metadata not found -> {station_name}")
        continue

    station_info = station_info.iloc[0]

    df = pd.read_excel(file)

    all_dfs.append((df, station_info))

print("\nLoaded Stations:", len(all_dfs))

Files found: 17

Loaded Stations: 17


# Build Unified Dataset

In [18]:
master = []

for df, station in all_dfs:

    df["datetime"] = (
        pd.to_datetime(df["Date"])
        + pd.to_timedelta(df["Hour"], unit="h")
    )

    temp = pd.DataFrame({

        "datetime": df["datetime"],

        "station_id": station["station_id"],
        "station_name": station["station_name"],

        "district": station["district"],

        "latitude": station["latitude"],
        "longitude": station["longitude"],

        "aqi": df["AQI"],

        "pm25": df["PM 2.5 AVG (µg/m³)"],
        "pm10": df["PM 10 AVG (µg/m³)"],

        "humidity": df["REL HUMI (%)"],
        "temperature": df["TEMPERATURE (°C)"],

        "wind_direction":
            df["Wind Direction Avg (°)"],

        "wind_speed_avg":
            df["Wind Speed Avg"],

        "wind_speed_min":
            df["Wind Speed Minimum (in m/s)"],

        "wind_speed_max":
            df["Wind Speed Maximum (in m/s)"]
    })

    master.append(temp)

master_df = pd.concat(
    master,
    ignore_index=True
)

master_df = (
    master_df
    .sort_values(["datetime", "station_id"])
    .reset_index(drop=True)
)

print(master_df.shape)
master_df.head()

(508116, 15)


,datetime,station_id,station_name,district,latitude,longitude,aqi,pm25,pm10,humidity,temperature,wind_direction,wind_speed_avg,wind_speed_min,wind_speed_max
0,2022-01-01 21:00:00,WB005,"East Calcutta Girls College, Lake Town",Kolkata,22.601583,88.404556,123,66.79,109.32,71.50,18.09,0,0.0,0.0,0.0
1,2022-01-01 21:00:00,WB009,Leather Complex,Kolkata,22.495300,88.509293,193,87.98,173.04,72.72,18.48,0,0.0,0.0,0.0
2,2022-01-01 21:00:00,WB015,Sarsuna College,Kolkata,22.481270,88.284554,185,85.59,194.13,73.95,18.51,0,0.0,0.0,0.0
3,2022-01-01 22:00:00,WB005,"East Calcutta Girls College, Lake Town",Kolkata,22.601583,88.404556,145,73.56,156.15,75.74,17.73,0,0.0,0.0,0.0
4,2022-01-01 22:00:00,WB009,Leather Complex,Kolkata,22.495300,88.509293,231,99.37,168.51,75.08,18.05,0,0.0,0.0,0.0


# Missing Values Check

In [19]:
missing = (
    master_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_percent = (
    master_df
    .isnull()
    .mean()
    .mul(100)
    .round(2)
)

quality = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent
})

quality

,missing_count,missing_percent
datetime,0,0.0
station_id,0,0.0
station_name,0,0.0
district,0,0.0
latitude,0,0.0
longitude,0,0.0
aqi,0,0.0
pm25,0,0.0
pm10,0,0.0
humidity,0,0.0


# Important Validation

In [20]:
print("=" * 50)

print("Rows:", len(master_df))

print("Stations:",
      master_df["station_id"].nunique())

print("Date Range:",
      master_df["datetime"].min(),
      "to",
      master_df["datetime"].max())

print("=" * 50)

Rows: 508116
Stations: 17
Date Range: 2022-01-01 21:00:00 to 2026-05-30 21:00:00


# Save Unified Dataset

In [21]:
output_path = "../../data/processed/wbpcb_master.parquet"

master_df.to_parquet(
    output_path,
    index=False
)

print(f"Saved successfully:\n{output_path}")

Saved successfully:
../../data/processed/wbpcb_master.parquet
